# Пара 14. Своё Streamlit-приложение с нуля

| | |
|---|---|
| **Зачем** | Научиться **самим** делать веб-приложения под свою диссертацию — не пользоваться чужой панелью, а собрать свой кабинет и дальше развивать его. |
| **Что получите** | Свой «Кабинет» на своих текстах: частоты, контексты, блокнот цитат с выгрузкой в Word, литература через API, **тренажёр по своим текстам** — и шаблон, как добавлять новые разделы с помощью бесплатного ИИ. |
| **Как** | 8 шагов. Каждый шаг — одна ячейка = весь файл приложения; новые строки помечены `# НОВОЕ`. Код целиком перед глазами, никаких скрытых библиотек. |

**Две пары:** A — шаги 1–5 (каркас кабинета на своих текстах); B — шаги 6–8 и свой раздел (шаг 9).

### Как работать
1. **Файл → Сохранить копию на Диске.** Подготовьте 2–5 своих текстов (.txt или .docx): главы, рассказы, стихотворения.
2. Запустите **«Подготовка»**, потом ячейку **Шага 1**, потом **«Запуск»** — откройте ссылку `…trycloudflare.com`.
3. Дальше просто запускайте ячейки шагов по очереди: каждая перезаписывает `my_app.py`, страница обновляется сама.
4. Свои правки вносите **в ячейку текущего шага**. Следующий шаг начинается с «чистой» версии — если хотите сохранить свои находки, перенесите их.
5. Ошибка на странице — не страшно: прочитайте последнюю строку красного блока, исправьте, запустите ячейку снова.

In [ ]:
#@title 🔧 Подготовка — установить Streamlit (≈1 мин) { display-mode: "form" }
import os, subprocess
subprocess.run("pip -q install streamlit python-docx", shell=True)
if not os.path.exists("cloudflared"):
    subprocess.run("wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared", shell=True)
print("✓ Готово. Теперь — ячейка Шага 1.")

## Шаг 1. Первая страница
Заголовок, поле для текста и счётчик слов. Запустите ячейку с кодом (она **сохраняет файл** `my_app.py`), затем — ячейку «Запуск» ниже. Откройте ссылку.

In [ ]:
%%writefile my_app.py
# -*- coding: utf-8 -*-
# ШАГ 1. Самое маленькое приложение: заголовок, поле для текста, подсчёт слов.
# Каждая команда st.… — это один элемент на веб-странице.
import streamlit as st

st.title("Мой кабинет: тексты диссертации")                     # заголовок страницы
st.write("Вставьте фрагмент текста — я посчитаю слова.")          # обычный текст

text = st.text_area("Текст", "Темнел московский серый зимний день…", height=150)  # поле ввода

words = text.split()                                              # обычный Python: делим по пробелам
st.metric("Слов во фрагменте", len(words))                        # крупная цифра

# ЗАДАНИЯ: 1) впишите в заголовок свою тему; 2) добавьте st.metric("Знаков", len(text));
# 3) добавьте st.text_input("Мой автор") и выведите его через st.write.

In [ ]:
#@title ▶ Запуск моего приложения my_app.py (один раз — дальше страница обновляется сама) { display-mode: "form" }
APP = "my_app.py"
import subprocess, time, socket, re
subprocess.run("pkill -f 'streamlit run'; pkill -f 'cloudflared tunnel'", shell=True)
time.sleep(1)
st_log = open("streamlit.log", "w")
subprocess.Popen(["streamlit", "run", APP, "--server.port", "8501", "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
                  "--server.runOnSave", "true", "--browser.gatherUsageStats", "false"],
                 stdout=st_log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        socket.create_connection(("127.0.0.1", 8501), 1).close(); break
    except OSError:
        time.sleep(1)
else:
    print("✗ Streamlit не запустился. Посмотрите журнал в ячейке ниже."); raise SystemExit
cf_log = open("cloudflared.log", "w")
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"], stdout=cf_log, stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        url = m.group(0); break
from IPython.display import HTML, display
if url:
    display(HTML(f'<p style="font-size:18px">✓ Приложение <b>{APP}</b> запущено: <a href="{url}" target="_blank">{url}</a></p>'
                 '<p>Ссылка открывается в любом браузере, в том числе на телефоне. Если страница не грузится — подождите 10–20 секунд и обновите.</p>'))
else:
    print("Туннель не ответил. Запасной вариант — откройте через Colab:")
    from google.colab import output
    output.serve_kernel_port_as_window(8501, anchor_text="▶ Открыть приложение")

In [ ]:
#@title 🩺 Журнал приложения (если страница не открывается или белая) { display-mode: "form" }
print(open("streamlit.log").read()[-3000:])

Правьте код **прямо в ячейке** и запускайте её снова (Shift+Enter) — страница приложения обновится сама (если нет — меню ⋮ → *Rerun*).

**Задания** — в конце кода ячейки (`# ЗАДАНИЯ`). Сделайте хотя бы одно, прежде чем идти дальше.

## Шаг 2. Свои файлы
Загружаем свой текст (.txt или .docx). Новые приёмы: `st.file_uploader`, `st.stop()` (остановить страницу, пока нет файла), `st.columns`, `st.expander`.

In [ ]:
%%writefile my_app.py
# -*- coding: utf-8 -*-
# ШАГ 2. Загружаем СВОИ файлы (.txt или .docx). Новые строки помечены «# НОВОЕ».
import streamlit as st


def read_file(f):                                                  # НОВОЕ: функция чтения файла
    data = f.getvalue()                                            # НОВОЕ: байты файла
    if f.name.lower().endswith(".docx"):                           # НОВОЕ: Word читаем через python-docx
        import io
        from docx import Document
        from docx.oxml.ns import qn  # НОВОЕ
        from docx.text.paragraph import Paragraph  # НОВОЕ
        doc = Document(io.BytesIO(data))  # НОВОЕ
        # все абзацы, включая таблицы и надписи (doc.paragraphs таблицы пропускает)
        return "\n".join(Paragraph(p, doc).text for p in doc.element.body.iter(qn("w:p")))  # НОВОЕ
    for enc in ("utf-8-sig", "cp1251"):                            # НОВОЕ: пробуем две кодировки
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            pass
    return data.decode("utf-8", errors="replace")


st.title("Мой кабинет: тексты диссертации")

f = st.file_uploader("Загрузите текст (.txt или .docx)", type=["txt", "docx"])   # НОВОЕ
if f is None:                                                      # НОВОЕ: файла ещё нет —
    st.info("Загрузите файл, чтобы начать.")                       # НОВОЕ: подсказка
    st.stop()                                                      # НОВОЕ: дальше код не выполняется

text = read_file(f)                                                # НОВОЕ
words = text.split()
c1, c2 = st.columns(2)                                             # НОВОЕ: две колонки рядом
c1.metric("Слов", len(words))
c2.metric("Знаков", len(text))

with st.expander("Показать весь текст"):                           # НОВОЕ: раскрывающийся блок
    st.text_area("Текст целиком (прокрутите)", text, height=400)   # НОВОЕ: весь текст, с прокруткой

# ЗАДАНИЯ: 1) загрузите главу/рассказ своего автора; 2) добавьте третью колонку «Абзацев»
# (text.count("\n") + 1); 3) проверьте, что в окне виден конец вашего текста.

**Задания** — в конце кода ячейки (`# ЗАДАНИЯ`). Сделайте хотя бы одно, прежде чем идти дальше.

## Шаг 3. Частотный словарь
Регулярное выражение режет текст на слова, `Counter` их считает, `pandas` делает таблицу. Новые элементы: ползунок, галочка, график, кнопка «Скачать».

In [ ]:
%%writefile my_app.py
# -*- coding: utf-8 -*-
# ШАГ 3. Частотный словарь: таблица, график, скачивание. Новые строки — «# НОВОЕ».
import re                                                          # НОВОЕ: регулярные выражения
from collections import Counter                                    # НОВОЕ: счётчик слов

import pandas as pd                                                # НОВОЕ: таблицы
import streamlit as st

STOP = set("и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее её мне было вот от меня еще ещё нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между это".split())  # НОВОЕ: стоп-слова


def read_file(f):
    data = f.getvalue()
    if f.name.lower().endswith(".docx"):
        import io
        from docx import Document
        from docx.oxml.ns import qn
        from docx.text.paragraph import Paragraph
        doc = Document(io.BytesIO(data))
        # все абзацы, включая таблицы и надписи (doc.paragraphs таблицы пропускает)
        return "\n".join(Paragraph(p, doc).text for p in doc.element.body.iter(qn("w:p")))
    for enc in ("utf-8-sig", "cp1251"):
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            pass
    return data.decode("utf-8", errors="replace")


def tokenize(text):                                                # НОВОЕ: текст → список слов
    return re.findall(r"[а-яёa-z]+(?:-[а-яёa-z]+)?", text.lower())


st.title("Мой кабинет: тексты диссертации")

f = st.file_uploader("Загрузите текст (.txt или .docx)", type=["txt", "docx"])
if f is None:
    st.info("Загрузите файл, чтобы начать.")
    st.stop()

text = read_file(f)
words = tokenize(text)                                             # НОВОЕ: теперь слова без знаков препинания
c1, c2 = st.columns(2)
c1.metric("Слов", len(words))
c2.metric("Разных слов", len(set(words)))

st.subheader("Частотный словарь")                                  # НОВОЕ
top_n = st.slider("Сколько слов показать", 10, 100, 30)            # НОВОЕ: ползунок
skip_stop = st.checkbox("Убрать служебные слова", value=True)      # НОВОЕ: галочка
counted = Counter(w for w in words if not (skip_stop and w in STOP))  # НОВОЕ
df = pd.DataFrame(counted.most_common(top_n), columns=["слово", "частота"])  # НОВОЕ
st.bar_chart(df.set_index("слово"))                                # НОВОЕ: график
st.dataframe(df, hide_index=True)                                  # НОВОЕ: таблица
st.download_button("⬇ Скачать CSV (Excel)", df.to_csv(index=False).encode("utf-8-sig"),
                   "chastoty.csv", "text/csv")                     # НОВОЕ: кнопка скачивания

# ЗАДАНИЯ: 1) найдите в списке 3 слова, важных для вашей темы; 2) добавьте в STOP
# слова, которые мешают; 3) добавьте колонку «на 1000 слов»: df["на 1000"] = df["частота"] * 1000 / len(words)

**Задания** — в конце кода ячейки (`# ЗАДАНИЯ`). Сделайте хотя бы одно, прежде чем идти дальше.

## Шаг 4. Контексты
Своя функция `kwic` — конкорданс: где стоит слово и что вокруг. Это главный инструмент доказательства в диссертации.

In [ ]:
%%writefile my_app.py
# -*- coding: utf-8 -*-
# ШАГ 4. Контексты (конкорданс): где и как стоит слово. Новые строки — «# НОВОЕ».
import re                                                          # регулярные выражения
from collections import Counter                                    # счётчик слов

import pandas as pd                                                # таблицы
import streamlit as st

STOP = set("и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее её мне было вот от меня еще ещё нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между это".split())  # стоп-слова


def read_file(f):
    data = f.getvalue()
    if f.name.lower().endswith(".docx"):
        import io
        from docx import Document
        from docx.oxml.ns import qn
        from docx.text.paragraph import Paragraph
        doc = Document(io.BytesIO(data))
        # все абзацы, включая таблицы и надписи (doc.paragraphs таблицы пропускает)
        return "\n".join(Paragraph(p, doc).text for p in doc.element.body.iter(qn("w:p")))
    for enc in ("utf-8-sig", "cp1251"):
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            pass
    return data.decode("utf-8", errors="replace")


def tokenize(text):                                                # текст → список слов
    return re.findall(r"[а-яёa-z]+(?:-[а-яёa-z]+)?", text.lower())


st.title("Мой кабинет: тексты диссертации")

f = st.file_uploader("Загрузите текст (.txt или .docx)", type=["txt", "docx"])
if f is None:
    st.info("Загрузите файл, чтобы начать.")
    st.stop()

text = read_file(f)
words = tokenize(text)                                             # теперь слова без знаков препинания
c1, c2 = st.columns(2)
c1.metric("Слов", len(words))
c2.metric("Разных слов", len(set(words)))

st.subheader("Частотный словарь")                                
top_n = st.slider("Сколько слов показать", 10, 100, 30)            # ползунок
skip_stop = st.checkbox("Убрать служебные слова", value=True)      # галочка
counted = Counter(w for w in words if not (skip_stop and w in STOP))
df = pd.DataFrame(counted.most_common(top_n), columns=["слово", "частота"])
st.bar_chart(df.set_index("слово"))                                # график
st.dataframe(df, hide_index=True)                                  # таблица
st.download_button("⬇ Скачать CSV (Excel)", df.to_csv(index=False).encode("utf-8-sig"),
                   "chastoty.csv", "text/csv")


def kwic(text, word, width=60):                                    # НОВОЕ: поиск контекстов
    rows = []                                                      # НОВОЕ
    for m in re.finditer(r"\b" + re.escape(word.lower()), text.lower()):  # НОВОЕ: каждое вхождение
        a, b = m.start(), m.end()                                  # НОВОЕ
        rows.append({"слева": text[max(0, a - width):a].replace("\n", " "),   # НОВОЕ
                     "слово": text[a:b],                           # НОВОЕ
                     "справа": text[b:b + width].replace("\n", " ")})       # НОВОЕ
    return pd.DataFrame(rows)                                      # НОВОЕ


st.subheader("Контексты")                                          # НОВОЕ
word = st.text_input("Слово или начало слова (вспомин — найдёт вспоминал, вспоминая…)", "")  # НОВОЕ
width = st.slider("Ширина контекста, знаков", 20, 150, 60)         # НОВОЕ
if word.strip():                                                   # НОВОЕ: только если что-то введено
    found = kwic(text, word.strip(), width)                        # НОВОЕ
    st.write(f"Найдено: {len(found)}")                             # НОВОЕ
    st.dataframe(found, hide_index=True, use_container_width=True) # НОВОЕ

# ЗАДАНИЯ: 1) найдите контексты ключевого слова своей темы; 2) выпишите 2 контекста,
# которые подтверждают вашу гипотезу, и 1 — который ей противоречит; 3) добавьте кнопку
# скачивания контекстов в CSV (по образцу кнопки частот).

**Задания** — в конце кода ячейки (`# ЗАДАНИЯ`). Сделайте хотя бы одно, прежде чем идти дальше.

## Шаг 5. Кабинет с меню
Меню разделов в левой панели (`st.sidebar`, `st.radio`) и **несколько** текстов сразу (`accept_multiple_files=True`). Код частот и контекстов тот же — разложен по разделам `if page == …`.

In [ ]:
%%writefile my_app.py
# -*- coding: utf-8 -*-
# ШАГ 5. Настоящий кабинет: меню разделов слева и НЕСКОЛЬКО своих текстов. Новые строки — «# НОВОЕ».
# Код частот и контекстов тот же, что в шагах 3–4, — просто разложен по разделам.
import re                                                          # регулярные выражения
from collections import Counter                                    # счётчик слов

import pandas as pd                                                # таблицы
import streamlit as st

STOP = set("и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее её мне было вот от меня еще ещё нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между это".split())  # стоп-слова


def read_file(f):
    data = f.getvalue()
    if f.name.lower().endswith(".docx"):
        import io
        from docx import Document
        from docx.oxml.ns import qn
        from docx.text.paragraph import Paragraph
        doc = Document(io.BytesIO(data))
        # все абзацы, включая таблицы и надписи (doc.paragraphs таблицы пропускает)
        return "\n".join(Paragraph(p, doc).text for p in doc.element.body.iter(qn("w:p")))
    for enc in ("utf-8-sig", "cp1251"):
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            pass
    return data.decode("utf-8", errors="replace")


def tokenize(text):                                                # текст → список слов
    return re.findall(r"[а-яёa-z]+(?:-[а-яёa-z]+)?", text.lower())


def kwic(text, word, width=60):                                    # поиск контекстов
    rows = []
    for m in re.finditer(r"\b" + re.escape(word.lower()), text.lower()):  # каждое вхождение
        a, b = m.start(), m.end()
        rows.append({"слева": text[max(0, a - width):a].replace("\n", " "),
                     "слово": text[a:b],
                     "справа": text[b:b + width].replace("\n", " ")})
    return pd.DataFrame(rows)



st.set_page_config(page_title="Мой кабинет", page_icon="📚", layout="wide")   # НОВОЕ: широкая страница

with st.sidebar:                                                   # НОВОЕ: всё внутри — в левой панели
    st.header("📚 Мой кабинет")                                     # НОВОЕ
    files = st.file_uploader("Мои тексты (.txt, .docx)", type=["txt", "docx"],
                             accept_multiple_files=True)           # НОВОЕ: можно выбрать много файлов
    page = st.radio("Раздел", ["Мои тексты", "Частоты", "Контексты"])  # НОВОЕ: меню

texts = {f.name.rsplit(".", 1)[0]: read_file(f) for f in files or []}  # НОВОЕ: {название: текст}
if not texts:
    st.info("Загрузите слева один или несколько файлов: главы, рассказы, стихотворения.")
    st.stop()

st.title(page)                                                     # НОВОЕ: заголовок = выбранный раздел

if page == "Мои тексты":                                           # НОВОЕ: раздел 1
    rows = [{"текст": name, "слов": len(tokenize(t)), "разных слов": len(set(tokenize(t)))}
            for name, t in texts.items()]                          # НОВОЕ: строка таблицы на каждый текст
    st.dataframe(pd.DataFrame(rows), hide_index=True)              # НОВОЕ
    name = st.selectbox("Прочитать", list(texts))                  # НОВОЕ: выпадающий список
    st.text_area("Текст целиком (прокрутите)", texts[name], height=400)

elif page == "Частоты":                                            # НОВОЕ: раздел 2
    name = st.selectbox("Текст", ["все вместе"] + list(texts))     # НОВОЕ
    text = "\n".join(texts.values()) if name == "все вместе" else texts[name]  # НОВОЕ
    words = tokenize(text)
    top_n = st.slider("Сколько слов показать", 10, 100, 30)
    skip_stop = st.checkbox("Убрать служебные слова", value=True)
    counted = Counter(w for w in words if not (skip_stop and w in STOP))
    df = pd.DataFrame(counted.most_common(top_n), columns=["слово", "частота"])
    st.bar_chart(df.set_index("слово"))
    st.dataframe(df, hide_index=True)
    st.download_button("⬇ Скачать CSV (Excel)", df.to_csv(index=False).encode("utf-8-sig"),
                       "chastoty.csv", "text/csv")

elif page == "Контексты":                                          # НОВОЕ: раздел 3
    word = st.text_input("Слово или начало слова", "")
    width = st.slider("Ширина контекста, знаков", 20, 150, 60)
    if word.strip():
        parts = []                                                 # НОВОЕ: ищем во всех текстах
        for name, t in texts.items():                              # НОВОЕ
            found = kwic(t, word.strip(), width)
            found.insert(0, "текст", name)                         # НОВОЕ: колонка «из какого текста»
            parts.append(found)
        found = pd.concat(parts, ignore_index=True)                # НОВОЕ: склеиваем таблицы
        st.write(f"Найдено: {len(found)}")
        st.dataframe(found, hide_index=True, use_container_width=True)

# ЗАДАНИЯ: 1) загрузите 3–5 своих текстов и сравните частоты; 2) в «Контекстах» найдите,
# в каком тексте ключевое слово встречается чаще; 3) добавьте в меню четвёртый пункт
# «О проекте» и напишите там (st.markdown) тему, материал и метод своей диссертации.

Это **каркас любого кабинета**: новый раздел = новый пункт меню + новый блок `elif page == …`.

**Задания** — в конце кода ячейки (`# ЗАДАНИЯ`). Сделайте хотя бы одно, прежде чем идти дальше.

### ✅ Конец пары A
**Сдать:** скриншот своего кабинета с 3+ своими текстами, раздел «Частоты» или «Контексты» с ключевым словом темы.

## Шаг 6. Мои цитаты
Приложение «помнит» записи между нажатиями: `st.session_state`. Форма `st.form` с кнопкой сохранения, выгрузка в **Word** (`python-docx`).

In [ ]:
%%writefile my_app.py
# -*- coding: utf-8 -*-
# ШАГ 6. Блокнот цитат: приложение «запоминает» ваши записи (st.session_state) и выгружает их в Word. Новые строки — «# НОВОЕ».
import re                                                          # регулярные выражения
from collections import Counter                                    # счётчик слов

import pandas as pd                                                # таблицы
import streamlit as st

STOP = set("и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее её мне было вот от меня еще ещё нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между это".split())  # стоп-слова


def read_file(f):
    data = f.getvalue()
    if f.name.lower().endswith(".docx"):
        import io
        from docx import Document
        from docx.oxml.ns import qn
        from docx.text.paragraph import Paragraph
        doc = Document(io.BytesIO(data))
        # все абзацы, включая таблицы и надписи (doc.paragraphs таблицы пропускает)
        return "\n".join(Paragraph(p, doc).text for p in doc.element.body.iter(qn("w:p")))
    for enc in ("utf-8-sig", "cp1251"):
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            pass
    return data.decode("utf-8", errors="replace")


def tokenize(text):                                                # текст → список слов
    return re.findall(r"[а-яёa-z]+(?:-[а-яёa-z]+)?", text.lower())


def kwic(text, word, width=60):                                    # поиск контекстов
    rows = []
    for m in re.finditer(r"\b" + re.escape(word.lower()), text.lower()):  # каждое вхождение
        a, b = m.start(), m.end()
        rows.append({"слева": text[max(0, a - width):a].replace("\n", " "),
                     "слово": text[a:b],
                     "справа": text[b:b + width].replace("\n", " ")})
    return pd.DataFrame(rows)



st.set_page_config(page_title="Мой кабинет", page_icon="📚", layout="wide")   # широкая страница

with st.sidebar:                                                   # всё внутри — в левой панели
    st.header("📚 Мой кабинет")
    files = st.file_uploader("Мои тексты (.txt, .docx)", type=["txt", "docx"],
                             accept_multiple_files=True)           # можно выбрать много файлов
    page = st.radio("Раздел", ["Мои тексты", "Частоты", "Контексты", "Мои цитаты"])  # НОВОЕ: + «Мои цитаты»

texts = {f.name.rsplit(".", 1)[0]: read_file(f) for f in files or []}  # {название: текст}
if not texts:
    st.info("Загрузите слева один или несколько файлов: главы, рассказы, стихотворения.")
    st.stop()

st.title(page)                                                     # заголовок = выбранный раздел

if page == "Мои тексты":                                           # раздел 1
    rows = [{"текст": name, "слов": len(tokenize(t)), "разных слов": len(set(tokenize(t)))}
            for name, t in texts.items()]                          # строка таблицы на каждый текст
    st.dataframe(pd.DataFrame(rows), hide_index=True)
    name = st.selectbox("Прочитать", list(texts))                  # выпадающий список
    st.text_area("Текст целиком (прокрутите)", texts[name], height=400)

elif page == "Частоты":                                            # раздел 2
    name = st.selectbox("Текст", ["все вместе"] + list(texts))
    text = "\n".join(texts.values()) if name == "все вместе" else texts[name]
    words = tokenize(text)
    top_n = st.slider("Сколько слов показать", 10, 100, 30)
    skip_stop = st.checkbox("Убрать служебные слова", value=True)
    counted = Counter(w for w in words if not (skip_stop and w in STOP))
    df = pd.DataFrame(counted.most_common(top_n), columns=["слово", "частота"])
    st.bar_chart(df.set_index("слово"))
    st.dataframe(df, hide_index=True)
    st.download_button("⬇ Скачать CSV (Excel)", df.to_csv(index=False).encode("utf-8-sig"),
                       "chastoty.csv", "text/csv")

elif page == "Контексты":                                          # раздел 3
    word = st.text_input("Слово или начало слова", "")
    width = st.slider("Ширина контекста, знаков", 20, 150, 60)
    if word.strip():
        parts = []                                                 # ищем во всех текстах
        for name, t in texts.items():
            found = kwic(t, word.strip(), width)
            found.insert(0, "текст", name)                         # колонка «из какого текста»
            parts.append(found)
        found = pd.concat(parts, ignore_index=True)                # склеиваем таблицы
        st.write(f"Найдено: {len(found)}")
        st.dataframe(found, hide_index=True, use_container_width=True)

elif page == "Мои цитаты":                                         # НОВОЕ: раздел 4
    if "quotes" not in st.session_state:                           # НОВОЕ: «память» приложения
        st.session_state.quotes = []                               # НОВОЕ: пустой список при первом запуске
    with st.form("new_quote", clear_on_submit=True):               # НОВОЕ: форма с кнопкой отправки
        quote = st.text_area("Цитата (скопируйте из «Контекстов» или из текста)")  # НОВОЕ
        source = st.selectbox("Из какого текста", list(texts))     # НОВОЕ
        note = st.text_input("Мой комментарий: что эта цитата доказывает")  # НОВОЕ
        if st.form_submit_button("Сохранить цитату") and quote.strip():     # НОВОЕ
            st.session_state.quotes.append({"текст": source, "цитата": quote.strip(), "комментарий": note})  # НОВОЕ
    q = pd.DataFrame(st.session_state.quotes)                      # НОВОЕ
    st.write(f"Сохранено цитат: {len(q)}")                         # НОВОЕ
    if not q.empty:                                                # НОВОЕ
        st.dataframe(q, hide_index=True, use_container_width=True) # НОВОЕ
        st.download_button("⬇ CSV", q.to_csv(index=False).encode("utf-8-sig"), "citaty.csv", "text/csv")  # НОВОЕ
        import io                                                  # НОВОЕ: выгрузка в Word
        from docx import Document                                  # НОВОЕ
        doc = Document()                                           # НОВОЕ: пустой документ
        doc.add_heading("Цитаты для диссертации", 1)               # НОВОЕ
        for _, r in q.iterrows():                                  # НОВОЕ: по одной цитате
            doc.add_paragraph(f"«{r['цитата']}» ({r['текст']})")   # НОВОЕ
            if r["комментарий"]:                                   # НОВОЕ
                doc.add_paragraph().add_run("Комментарий: " + r["комментарий"]).italic = True  # НОВОЕ: курсивом
        buf = io.BytesIO()                                         # НОВОЕ
        doc.save(buf)                                              # НОВОЕ
        st.download_button("⬇ Word (.docx)", buf.getvalue(), "citaty.docx")  # НОВОЕ
    st.warning("Цитаты живут, пока открыта вкладка и работает Colab. Скачивайте CSV/Word в конце работы!")  # НОВОЕ

# ЗАДАНИЯ: 1) сохраните 5 цитат по своей теме с комментариями и скачайте Word;
# 2) добавьте в форму поле «Глава диссертации» (st.selectbox с вариантами «Глава 1», «Глава 2»…)
# и выведите его в таблицу и в Word; 3) добавьте кнопку st.button("Очистить всё"),
# которая делает st.session_state.quotes = [].

**Задания** — в конце кода ячейки (`# ЗАДАНИЯ`). Сделайте хотя бы одно, прежде чем идти дальше.

## Шаг 7. Литература через API
Запрос к OpenAlex через `requests`, ответ в JSON → таблица. `@st.cache_data` запоминает ответ, чтобы не дёргать сервер при каждом нажатии.

In [ ]:
%%writefile my_app.py
# -*- coding: utf-8 -*-
# ШАГ 7. Подключаем API: литература по теме из OpenAlex (st.cache_data — не спрашивать сервер дважды). Новые строки — «# НОВОЕ».
import re                                                          # регулярные выражения
from collections import Counter                                    # счётчик слов

import requests                                                    # НОВОЕ: запросы к API
import pandas as pd                                                # таблицы
import streamlit as st

STOP = set("и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее её мне было вот от меня еще ещё нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между это".split())  # стоп-слова


def read_file(f):
    data = f.getvalue()
    if f.name.lower().endswith(".docx"):
        import io
        from docx import Document
        from docx.oxml.ns import qn
        from docx.text.paragraph import Paragraph
        doc = Document(io.BytesIO(data))
        # все абзацы, включая таблицы и надписи (doc.paragraphs таблицы пропускает)
        return "\n".join(Paragraph(p, doc).text for p in doc.element.body.iter(qn("w:p")))
    for enc in ("utf-8-sig", "cp1251"):
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            pass
    return data.decode("utf-8", errors="replace")


def tokenize(text):                                                # текст → список слов
    return re.findall(r"[а-яёa-z]+(?:-[а-яёa-z]+)?", text.lower())


def kwic(text, word, width=60):                                    # поиск контекстов
    rows = []
    for m in re.finditer(r"\b" + re.escape(word.lower()), text.lower()):  # каждое вхождение
        a, b = m.start(), m.end()
        rows.append({"слева": text[max(0, a - width):a].replace("\n", " "),
                     "слово": text[a:b],
                     "справа": text[b:b + width].replace("\n", " ")})
    return pd.DataFrame(rows)



@st.cache_data(ttl=3600)                                           # НОВОЕ: запомнить ответ на час
def openalex(query, n=25):                                         # НОВОЕ: запрос к API
    r = requests.get("https://api.openalex.org/works",             # НОВОЕ: адрес API
                     params={"search": query, "per_page": n, "sort": "cited_by_count:desc"},  # НОВОЕ
                     timeout=30)                                   # НОВОЕ
    r.raise_for_status()                                           # НОВОЕ: ошибка сервера → исключение
    return r.json()                                                # НОВОЕ: ответ в формате JSON → словарь

st.set_page_config(page_title="Мой кабинет", page_icon="📚", layout="wide")   # широкая страница

with st.sidebar:                                                   # всё внутри — в левой панели
    st.header("📚 Мой кабинет")
    files = st.file_uploader("Мои тексты (.txt, .docx)", type=["txt", "docx"],
                             accept_multiple_files=True)           # можно выбрать много файлов
    page = st.radio("Раздел", ["Мои тексты", "Частоты", "Контексты", "Мои цитаты", "Литература"])  # НОВОЕ

texts = {f.name.rsplit(".", 1)[0]: read_file(f) for f in files or []}  # {название: текст}
if not texts:
    st.info("Загрузите слева один или несколько файлов: главы, рассказы, стихотворения.")
    st.stop()

st.title(page)                                                     # заголовок = выбранный раздел

if page == "Мои тексты":                                           # раздел 1
    rows = [{"текст": name, "слов": len(tokenize(t)), "разных слов": len(set(tokenize(t)))}
            for name, t in texts.items()]                          # строка таблицы на каждый текст
    st.dataframe(pd.DataFrame(rows), hide_index=True)
    name = st.selectbox("Прочитать", list(texts))                  # выпадающий список
    st.text_area("Текст целиком (прокрутите)", texts[name], height=400)

elif page == "Частоты":                                            # раздел 2
    name = st.selectbox("Текст", ["все вместе"] + list(texts))
    text = "\n".join(texts.values()) if name == "все вместе" else texts[name]
    words = tokenize(text)
    top_n = st.slider("Сколько слов показать", 10, 100, 30)
    skip_stop = st.checkbox("Убрать служебные слова", value=True)
    counted = Counter(w for w in words if not (skip_stop and w in STOP))
    df = pd.DataFrame(counted.most_common(top_n), columns=["слово", "частота"])
    st.bar_chart(df.set_index("слово"))
    st.dataframe(df, hide_index=True)
    st.download_button("⬇ Скачать CSV (Excel)", df.to_csv(index=False).encode("utf-8-sig"),
                       "chastoty.csv", "text/csv")

elif page == "Контексты":                                          # раздел 3
    word = st.text_input("Слово или начало слова", "")
    width = st.slider("Ширина контекста, знаков", 20, 150, 60)
    if word.strip():
        parts = []                                                 # ищем во всех текстах
        for name, t in texts.items():
            found = kwic(t, word.strip(), width)
            found.insert(0, "текст", name)                         # колонка «из какого текста»
            parts.append(found)
        found = pd.concat(parts, ignore_index=True)                # склеиваем таблицы
        st.write(f"Найдено: {len(found)}")
        st.dataframe(found, hide_index=True, use_container_width=True)

elif page == "Мои цитаты":                                         # раздел 4
    if "quotes" not in st.session_state:                           # «память» приложения
        st.session_state.quotes = []                               # пустой список при первом запуске
    with st.form("new_quote", clear_on_submit=True):               # форма с кнопкой отправки
        quote = st.text_area("Цитата (скопируйте из «Контекстов» или из текста)")
        source = st.selectbox("Из какого текста", list(texts))
        note = st.text_input("Мой комментарий: что эта цитата доказывает")
        if st.form_submit_button("Сохранить цитату") and quote.strip():
            st.session_state.quotes.append({"текст": source, "цитата": quote.strip(), "комментарий": note})
    q = pd.DataFrame(st.session_state.quotes)
    st.write(f"Сохранено цитат: {len(q)}")
    if not q.empty:
        st.dataframe(q, hide_index=True, use_container_width=True)
        st.download_button("⬇ CSV", q.to_csv(index=False).encode("utf-8-sig"), "citaty.csv", "text/csv")
        import io                                                  # выгрузка в Word
        from docx import Document
        doc = Document()                                           # пустой документ
        doc.add_heading("Цитаты для диссертации", 1)
        for _, r in q.iterrows():                                  # по одной цитате
            doc.add_paragraph(f"«{r['цитата']}» ({r['текст']})")
            if r["комментарий"]:
                doc.add_paragraph().add_run("Комментарий: " + r["комментарий"]).italic = True  # курсивом
        buf = io.BytesIO()
        doc.save(buf)
        st.download_button("⬇ Word (.docx)", buf.getvalue(), "citaty.docx")
    st.warning("Цитаты живут, пока открыта вкладка и работает Colab. Скачивайте CSV/Word в конце работы!")

elif page == "Литература":                                         # НОВОЕ: раздел 5
    st.caption("OpenAlex — открытый каталог научных работ. Запрос лучше по-английски, 2–4 слова.")  # НОВОЕ
    query = st.text_input("Запрос", "Bunin memory")                # НОВОЕ
    if query.strip():                                              # НОВОЕ
        try:                                                       # НОВОЕ: сеть может подвести
            data = openalex(query.strip())                         # НОВОЕ
        except Exception as e:                                     # НОВОЕ
            st.error(f"OpenAlex не ответил: {e}. Подождите минуту и попробуйте снова.")  # НОВОЕ
            st.stop()                                              # НОВОЕ
        rows = [{"год": w.get("publication_year"),                 # НОВОЕ: из JSON берём нужные поля
                 "название": w.get("display_name"),                # НОВОЕ
                 "авторы": ", ".join(a["author"]["display_name"] for a in w.get("authorships", [])[:3]),  # НОВОЕ
                 "цитирований": w.get("cited_by_count"),           # НОВОЕ
                 "DOI": w.get("doi") or ""}                        # НОВОЕ
                for w in data.get("results", [])]                  # НОВОЕ
        lit = pd.DataFrame(rows)                                   # НОВОЕ
        st.write(f"Всего в базе: {data.get('meta', {}).get('count', 0)}, показано: {len(lit)}")  # НОВОЕ
        if not lit.empty:                                          # НОВОЕ
            st.dataframe(lit, hide_index=True, use_container_width=True,
                         column_config={"DOI": st.column_config.LinkColumn()})  # НОВОЕ: ссылки кликабельны
            st.bar_chart(lit["год"].value_counts().sort_index())   # НОВОЕ: работы по годам
            st.download_button("⬇ CSV", lit.to_csv(index=False).encode("utf-8-sig"), "literatura.csv", "text/csv")  # НОВОЕ

# ЗАДАНИЯ: 1) подберите запрос по своей теме; 2) добавьте ползунок «годы от–до» и
# фильтр в params: "filter": f"from_publication_date:{y1}-01-01,to_publication_date:{y2}-12-31";
# 3) добавьте колонку «журнал»: ((w.get("primary_location") or {}).get("source") or {}).get("display_name").

**Задания** — в конце кода ячейки (`# ЗАДАНИЯ`). Сделайте хотя бы одно, прежде чем идти дальше.

## Шаг 8. Тренажёр по своим текстам
Случайное предложение из **ваших** текстов: угадать пропущенное слово или угадать, из какого текста фраза. `random`, задание хранится в `st.session_state`.

In [ ]:
%%writefile my_app.py
# -*- coding: utf-8 -*-
# ШАГ 8. Тренажёр по СВОИМ текстам: угадай пропущенное слово / угадай, из какого текста фраза. Новые строки — «# НОВОЕ».
import re                                                          # регулярные выражения
from collections import Counter                                    # счётчик слов

import requests                                                    # запросы к API
import random                                                     # НОВОЕ: случайный выбор
import pandas as pd                                                # таблицы
import streamlit as st

STOP = set("и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее её мне было вот от меня еще ещё нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между это".split())  # стоп-слова


def read_file(f):
    data = f.getvalue()
    if f.name.lower().endswith(".docx"):
        import io
        from docx import Document
        from docx.oxml.ns import qn
        from docx.text.paragraph import Paragraph
        doc = Document(io.BytesIO(data))
        # все абзацы, включая таблицы и надписи (doc.paragraphs таблицы пропускает)
        return "\n".join(Paragraph(p, doc).text for p in doc.element.body.iter(qn("w:p")))
    for enc in ("utf-8-sig", "cp1251"):
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            pass
    return data.decode("utf-8", errors="replace")


def tokenize(text):                                                # текст → список слов
    return re.findall(r"[а-яёa-z]+(?:-[а-яёa-z]+)?", text.lower())


def kwic(text, word, width=60):                                    # поиск контекстов
    rows = []
    for m in re.finditer(r"\b" + re.escape(word.lower()), text.lower()):  # каждое вхождение
        a, b = m.start(), m.end()
        rows.append({"слева": text[max(0, a - width):a].replace("\n", " "),
                     "слово": text[a:b],
                     "справа": text[b:b + width].replace("\n", " ")})
    return pd.DataFrame(rows)



@st.cache_data(ttl=3600)                                           # запомнить ответ на час
def openalex(query, n=25):                                         # запрос к API
    r = requests.get("https://api.openalex.org/works",             # адрес API
                     params={"search": query, "per_page": n, "sort": "cited_by_count:desc"},
                     timeout=30)
    r.raise_for_status()                                           # ошибка сервера → исключение
    return r.json()                                                # ответ в формате JSON → словарь

def sentences(text):                                               # НОВОЕ: текст → предложения
    parts = re.split(r"(?<=[.!?…])\s+", text.replace("\n", " "))  # НОВОЕ
    return [p.strip() for p in parts if len(p.split()) >= 8]       # НОВОЕ: только не слишком короткие


def make_task(texts, mode):                                        # НОВОЕ: составить задание
    pool = [(name, s) for name, t in texts.items() for s in sentences(t)]  # НОВОЕ: все предложения
    if mode == "Пропущенное слово":                                # НОВОЕ: нужны предложения с длинными словами
        pool = [(n, s) for n, s in pool if re.search(r"[А-Яа-яЁё]{5,}", s)]  # НОВОЕ
    if not pool:                                                   # НОВОЕ
        return None                                                # НОВОЕ
    name, sent = random.choice(pool)                               # НОВОЕ: случайное предложение
    task = {"id": str(random.random()), "mode": mode, "full": sent, "source": name}  # НОВОЕ
    if mode == "Пропущенное слово":                                # НОВОЕ
        word = random.choice(re.findall(r"[А-Яа-яЁё]{5,}", sent))  # НОВОЕ: случайное слово от 5 букв
        task.update(question=sent.replace(word, "_____", 1), answer=word)  # НОВОЕ
    else:                                                          # НОВОЕ
        task.update(question=sent, answer=name)                    # НОВОЕ
    return task                                                    # НОВОЕ

st.set_page_config(page_title="Мой кабинет", page_icon="📚", layout="wide")   # широкая страница

with st.sidebar:                                                   # всё внутри — в левой панели
    st.header("📚 Мой кабинет")
    files = st.file_uploader("Мои тексты (.txt, .docx)", type=["txt", "docx"],
                             accept_multiple_files=True)           # можно выбрать много файлов
    page = st.radio("Раздел", ["Мои тексты", "Частоты", "Контексты", "Мои цитаты", "Литература", "Тренажёр"])  # НОВОЕ

texts = {f.name.rsplit(".", 1)[0]: read_file(f) for f in files or []}  # {название: текст}
if not texts:
    st.info("Загрузите слева один или несколько файлов: главы, рассказы, стихотворения.")
    st.stop()

st.title(page)                                                     # заголовок = выбранный раздел

if page == "Мои тексты":                                           # раздел 1
    rows = [{"текст": name, "слов": len(tokenize(t)), "разных слов": len(set(tokenize(t)))}
            for name, t in texts.items()]                          # строка таблицы на каждый текст
    st.dataframe(pd.DataFrame(rows), hide_index=True)
    name = st.selectbox("Прочитать", list(texts))                  # выпадающий список
    st.text_area("Текст целиком (прокрутите)", texts[name], height=400)

elif page == "Частоты":                                            # раздел 2
    name = st.selectbox("Текст", ["все вместе"] + list(texts))
    text = "\n".join(texts.values()) if name == "все вместе" else texts[name]
    words = tokenize(text)
    top_n = st.slider("Сколько слов показать", 10, 100, 30)
    skip_stop = st.checkbox("Убрать служебные слова", value=True)
    counted = Counter(w for w in words if not (skip_stop and w in STOP))
    df = pd.DataFrame(counted.most_common(top_n), columns=["слово", "частота"])
    st.bar_chart(df.set_index("слово"))
    st.dataframe(df, hide_index=True)
    st.download_button("⬇ Скачать CSV (Excel)", df.to_csv(index=False).encode("utf-8-sig"),
                       "chastoty.csv", "text/csv")

elif page == "Контексты":                                          # раздел 3
    word = st.text_input("Слово или начало слова", "")
    width = st.slider("Ширина контекста, знаков", 20, 150, 60)
    if word.strip():
        parts = []                                                 # ищем во всех текстах
        for name, t in texts.items():
            found = kwic(t, word.strip(), width)
            found.insert(0, "текст", name)                         # колонка «из какого текста»
            parts.append(found)
        found = pd.concat(parts, ignore_index=True)                # склеиваем таблицы
        st.write(f"Найдено: {len(found)}")
        st.dataframe(found, hide_index=True, use_container_width=True)

elif page == "Мои цитаты":                                         # раздел 4
    if "quotes" not in st.session_state:                           # «память» приложения
        st.session_state.quotes = []                               # пустой список при первом запуске
    with st.form("new_quote", clear_on_submit=True):               # форма с кнопкой отправки
        quote = st.text_area("Цитата (скопируйте из «Контекстов» или из текста)")
        source = st.selectbox("Из какого текста", list(texts))
        note = st.text_input("Мой комментарий: что эта цитата доказывает")
        if st.form_submit_button("Сохранить цитату") and quote.strip():
            st.session_state.quotes.append({"текст": source, "цитата": quote.strip(), "комментарий": note})
    q = pd.DataFrame(st.session_state.quotes)
    st.write(f"Сохранено цитат: {len(q)}")
    if not q.empty:
        st.dataframe(q, hide_index=True, use_container_width=True)
        st.download_button("⬇ CSV", q.to_csv(index=False).encode("utf-8-sig"), "citaty.csv", "text/csv")
        import io                                                  # выгрузка в Word
        from docx import Document
        doc = Document()                                           # пустой документ
        doc.add_heading("Цитаты для диссертации", 1)
        for _, r in q.iterrows():                                  # по одной цитате
            doc.add_paragraph(f"«{r['цитата']}» ({r['текст']})")
            if r["комментарий"]:
                doc.add_paragraph().add_run("Комментарий: " + r["комментарий"]).italic = True  # курсивом
        buf = io.BytesIO()
        doc.save(buf)
        st.download_button("⬇ Word (.docx)", buf.getvalue(), "citaty.docx")
    st.warning("Цитаты живут, пока открыта вкладка и работает Colab. Скачивайте CSV/Word в конце работы!")

elif page == "Литература":                                         # раздел 5
    st.caption("OpenAlex — открытый каталог научных работ. Запрос лучше по-английски, 2–4 слова.")
    query = st.text_input("Запрос", "Bunin memory")
    if query.strip():
        try:                                                       # сеть может подвести
            data = openalex(query.strip())
        except Exception as e:
            st.error(f"OpenAlex не ответил: {e}. Подождите минуту и попробуйте снова.")
            st.stop()
        rows = [{"год": w.get("publication_year"),                 # из JSON берём нужные поля
                 "название": w.get("display_name"),
                 "авторы": ", ".join(a["author"]["display_name"] for a in w.get("authorships", [])[:3]),
                 "цитирований": w.get("cited_by_count"),
                 "DOI": w.get("doi") or ""}
                for w in data.get("results", [])]
        lit = pd.DataFrame(rows)
        st.write(f"Всего в базе: {data.get('meta', {}).get('count', 0)}, показано: {len(lit)}")
        if not lit.empty:
            st.dataframe(lit, hide_index=True, use_container_width=True,
                         column_config={"DOI": st.column_config.LinkColumn()})  # ссылки кликабельны
            st.bar_chart(lit["год"].value_counts().sort_index())   # работы по годам
            st.download_button("⬇ CSV", lit.to_csv(index=False).encode("utf-8-sig"), "literatura.csv", "text/csv")

elif page == "Тренажёр":                                           # НОВОЕ: раздел 6
    mode = st.radio("Режим", ["Пропущенное слово", "Из какого текста?"], horizontal=True)  # НОВОЕ
    if mode == "Из какого текста?" and len(texts) < 2:             # НОВОЕ
        st.info("Для этого режима загрузите хотя бы два текста.")  # НОВОЕ
        st.stop()                                                  # НОВОЕ
    if "task" not in st.session_state or st.session_state.task["mode"] != mode:  # НОВОЕ
        st.session_state.task = make_task(texts, mode)             # НОВОЕ: первое задание
    if st.button("🎲 Новое задание"):                              # НОВОЕ
        st.session_state.task = make_task(texts, mode)             # НОВОЕ
    task = st.session_state.task                                   # НОВОЕ
    if task is None:                                               # НОВОЕ
        st.warning("В текстах не нашлось подходящих предложений (нужны предложения от 8 слов).")  # НОВОЕ
        st.stop()                                                  # НОВОЕ
    st.markdown(f"> {task['question']}")                           # НОВОЕ: показываем задание цитатой
    if mode == "Пропущенное слово":                                # НОВОЕ
        guess = st.text_input("Какое слово пропущено?", key="guess_" + task["id"])  # НОВОЕ
        ok = guess.strip().lower() == task["answer"].lower()       # НОВОЕ
    else:                                                          # НОВОЕ
        guess = st.radio("Из какого текста?", list(texts), index=None, key="guess_" + task["id"])  # НОВОЕ
        ok = guess == task["answer"]                               # НОВОЕ
    if guess:                                                      # НОВОЕ: ответ дан — проверяем
        if ok:                                                     # НОВОЕ
            st.success("Верно! 🎉")                                # НОВОЕ
            if st.session_state.get("last") != task["id"]:         # НОВОЕ: не засчитывать одно задание дважды
                st.session_state.score = st.session_state.get("score", 0) + 1  # НОВОЕ
                st.session_state.last = task["id"]                 # НОВОЕ
        else:                                                      # НОВОЕ
            st.error(f"Нет. Правильно: {task['answer']}")          # НОВОЕ
        st.caption(f"Полностью: {task['full']} ({task['source']})")  # НОВОЕ
    st.metric("Верных ответов", st.session_state.get("score", 0))  # НОВОЕ

# ЗАДАНИЯ: 1) потренируйтесь на своих текстах: что запоминается, а что нет?
# 2) пусть пропускается не любое слово, а слово из ВАШЕГО списка (ключевые слова темы):
# KEY = ["память", "вспомнить"] и в make_task выбирайте только такие слова;
# 3) придумайте третий режим (например, «угадай героя» или «угадай главу»).
#
# ДАЛЬШЕ — САМИ: скопируйте весь этот файл в бесплатный чат ИИ (DeepSeek, Qwen, Gemini)
# и попросите добавить раздел — шаблон запроса есть в блокноте (шаг 9).

**Задания** — в конце кода ячейки (`# ЗАДАНИЯ`). Сделайте хотя бы одно, прежде чем идти дальше.

## Шаг 9. Свой раздел — вместе с бесплатным ИИ
Теперь вы знаете устройство: **меню → `elif page == …` → виджеты → вывод**. Новые разделы можно добавлять самим, а сложные места — просить у ИИ (DeepSeek, Qwen, Gemini — бесплатно). Шаблон запроса:

```
Я филолог-магистрант, учусь программировать. Вот мой файл Streamlit-приложения:
<вставьте сюда ВЕСЬ код my_app.py>

Добавь в меню новый раздел «……», который делает: ……
Требования: не меняй существующие разделы; используй только streamlit, pandas,
requests, python-docx и стандартную библиотеку Python; пометь новые строки
комментарием # НОВОЕ и объясни каждую одной фразой по-русски.
```
Ответ ИИ вставьте в новую ячейку, первой строкой оставьте `%%writefile my_app.py`. Ошибка — скопируйте её последнюю строку тому же ИИ: «вот ошибка, исправь и объясни».

**Идеи разделов для диссертации:**
- **Словарь терминов** моей работы (термин — определение — источник), выгрузка в Word.
- **Карточки для предзащиты**: вопрос → ответ, режим «перевернуть карточку».
- **Сравнение двух текстов** (оригинал и перевод, две редакции): частоты рядом, общие и уникальные слова.
- **Мои смысловые поля**: свой список слов по группам → частоты по текстам на 1000 слов.
- **Хронология**: события жизни автора / публикации (таблица + `st.line_chart`).
- **План глав**: глава — задача — материал — готовность (%), прогресс-бар `st.progress`.
- **Тренажёр «угадай героя»**: фраза героя → кто сказал.

**Сдать:** свой `my_app.py` с минимум одним собственным разделом (ячейка ниже) + одно предложение: что этот раздел даёт диссертации.

In [ ]:
#@title ⬇ Скачать своё приложение my_app.py { display-mode: "form" }
from google.colab import files
files.download("my_app.py")

## Шпаргалка Streamlit
| Нужно | Команда |
|---|---|
| заголовок / текст | `st.title("…")`, `st.subheader("…")`, `st.write(…)`, `st.markdown("**жирный**")` |
| поле ввода | `st.text_input("…")`, `st.text_area("…")`, `st.number_input("…", 1, 100)` |
| выбор | `st.selectbox("…", список)`, `st.radio("…", список)`, `st.multiselect(…)`, `st.checkbox("…")`, `st.slider("…", 0, 100, 10)` |
| файлы | `st.file_uploader("…", type=["txt"], accept_multiple_files=True)`, `st.download_button("…", данные, "имя.csv")` |
| вывод данных | `st.dataframe(df)`, `st.metric("…", 42)`, `st.bar_chart(df)`, `st.line_chart(df)`, `st.json(d)` |
| сообщения | `st.info`, `st.success`, `st.warning`, `st.error` |
| раскладка | `st.sidebar`, `c1, c2 = st.columns(2)`, `with st.expander("…"):`, `tab1, tab2 = st.tabs(["А", "Б"])` |
| кнопки и формы | `if st.button("…"):`, `with st.form("f"):` … `st.form_submit_button("…")` |
| память | `st.session_state.x = …` (живёт, пока открыта вкладка) |
| ускорение | `@st.cache_data` над функцией, которая ходит в интернет или долго считает |
| остановить страницу | `st.stop()` |

Главное правило: **при любом действии пользователя файл выполняется заново сверху вниз.** Всё, что должно «запомниться», кладите в `st.session_state`.

### По желанию: постоянная ссылка (Streamlit Community Cloud, бесплатно)
1. Зарегистрируйтесь на github.com, создайте репозиторий, загрузите туда `my_app.py` и `requirements.txt` со строками `streamlit`, `pandas`, `requests`, `python-docx`.
2. share.streamlit.io → вход через GitHub → **Create app** → репозиторий и файл `my_app.py` → **Deploy**.
Тексты, защищённые авторским правом, в публичный репозиторий не кладите — приложение загружает их через «Мои тексты».